# Ride Smart

O projeto RideSmart aborda o cenário em que um usuário aceita caminhar uma
distância máxima X a partir de uma origem A até um ponto de
embarque otimizado P, visando mitigar congestionamentos,
contornar barreiras geométricas ou acessar vias de tráfego
rápido, reduzindo o tempo ou a distância total de deslocamento
até o destino final B.

### Integrantes

- Henrique Eduardo Costa Da Silva
- João Victor Moura Lucas da Silva
- Murilo de Lima Barros
- Ramon Vinícius Ferreira de Souza

### Preparação do Ambiente

In [6]:
# Caso queira instalar as dependências diretamente no kernel, descomente a linha abaixo e execute o código
!pip install pandas geopandas osmnx matplotlib scikit-learn folium --quiet

In [ ]:
# Importa o caminho do projeto para o sys.path, permitindo a importação de módulos do projeto
import sys
import os

CURRENT_DIR = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(CURRENT_DIR, 'src')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print('src added to sys.path:', SRC_PATH)

### Carregando o grafo e preparando as estruturas
Baixa o grafo OSMnx e cria a lista de adjacência para a cidade de Natal, RN.

In [ ]:
import argparse
import osmnx as ox
from main import load_graph, build_adjacency_from_graph, generate_random_points, create_solver, save_animation, select_best_boarding_node
from visualization import animate_algorithm, create_folium_map

# Configuração: ajustar 'place' ou usar graphml local
args = argparse.Namespace(
    place='Natal, Brazil',
    graphml=None,
    origin_lat=-5.835069, # UFRN, Natal
    origin_lon=-35.2113248,
    dest_lat=-5.81116, # Midway Mall, Natal
    dest_lon=-35.2063,
    walk_radius=200.0,
    candidates=20,
    algorithm='dijkstra_heap',
    output=None,
    min_angle=0.0,
    seed=42,
    traffic='off'
)

print('Loading graph (may take time)')
G = load_graph(args)
G = ox.distance.add_edge_lengths(G)
adjacency, node_to_idx, idx_to_node, coords = build_adjacency_from_graph(G, weight_attr='length')
print('Nodes:', len(adjacency))

### Rodando  o Algoritmo

In [ ]:
# Encontra nós e gera candidatos
origin_node = ox.distance.nearest_nodes(G, X=args.origin_lon, Y=args.origin_lat)
dest_node = ox.distance.nearest_nodes(G, X=args.dest_lon, Y=args.dest_lat)
dest_idx = node_to_idx[dest_node]

candidates = generate_random_points(
    args.origin_lat, args.origin_lon, args.walk_radius, args.candidates, args.min_angle, G, seed=args.seed
)
candidate_nodes = list(dict.fromkeys(candidates))
candidate_indices = [node_to_idx[n] for n in candidate_nodes]
print('Candidatos gerados:', len(candidate_nodes))

solver = create_solver(args.algorithm, adjacency, coords)
html = animate_algorithm(G, solver, origin_node, dest_node, dest_idx, candidate_indices, idx_to_node, args.walk_radius)
display(html)

## Calcular melhor ponto e mostrar mapa Folium com caminho
Avalia cada candidato com o solver (já criado) para encontrar o melhor embarque e desenha o caminho no mapa Folium embutido.

In [ ]:
# Avaliar candidatos e obter melhor rota
best_node, best_weight, best_path = select_best_boarding_node(candidate_nodes, node_to_idx, solver, dest_idx)
print('Melhor nó de embarque:', best_node)

# Gerar e salvar mapa Folium com o melhor caminho
folium_out = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mapa_interativo.html')
os.makedirs(os.path.dirname(folium_out), exist_ok=True)
create_folium_map(G, origin_node, dest_node, candidate_nodes, best_path, idx_to_node, output_path=folium_out)
print('Mapa interativo salvo em', folium_out)

# Exibir mapa inline usando IFrame
from IPython.display import IFrame, display
display(IFrame(folium_out, width=900, height=600))